# FIT5196 Assessment 2 - Shared Foundations

**Group:** Group018  
**Stage:** Shared configuration, investigation and reusable business-rule functions

This notebook implements the common foundations needed by the dirty-data, missing-data and
outlier work. It follows the opening structure of the supplied solution template, but it is a
separate working notebook. It does **not** repair records, remove outliers, or export cleaned
CSV files.

The source files under `data/` are treated as immutable. They are loaded into separate source
DataFrames, copied for snapshot checks, and verified by SHA-256 hashes at the beginning and end.

## 0. Configuration and environment

This section keeps identifiers, paths, expected columns, tolerances and the random seed in one
place. Central configuration makes the workflow easier to reproduce and prevents later team
members from embedding different paths or thresholds in their own code.

The notebook expects to run from the assignment directory. It reads only from `data/` and does
not create cleaning outputs in this stage.

In [1]:
from pathlib import Path

GROUP_ID = "Group018"
DATA_DIR = Path("data")
RANDOM_SEED = 5196

DIRTY_INPUT = DATA_DIR / f"{GROUP_ID}_dirty_data.csv"
MISSING_INPUT = DATA_DIR / f"{GROUP_ID}_missing_data.csv"
OUTLIER_INPUT = DATA_DIR / f"{GROUP_ID}_outlier_data.csv"
WAREHOUSE_INPUT = DATA_DIR / "warehouses.csv"
MANIFEST_INPUT = DATA_DIR / "A2_manifest.json"
RESOURCE_CONTRACT_INPUT = DATA_DIR / "resource_contract.json"
VADER_LEXICON_INPUT = DATA_DIR / "resources/nltk_data/sentiment/vader_lexicon.zip"

EXPECTED_COLUMNS = [
    "order_id",
    "customer_id",
    "date",
    "nearest_warehouse",
    "shopping_cart",
    "order_price",
    "delivery_charges",
    "customer_lat",
    "customer_long",
    "coupon_discount",
    "order_total",
    "season",
    "is_expedited_delivery",
    "distance_to_nearest_warehouse",
    "latest_customer_review",
    "is_happy_customer",
]

EXPECTED_ROWS_PER_BATCH = 500
MONEY_TOLERANCE = 0.01
DISTANCE_TOLERANCE = 0.0001
EARTH_RADIUS_KM = 6371.0
NO_REVIEW_SENTINEL = "No review submitted"

### 0.1 Dependencies and frozen local resources

The assignment requires NLTK 3.9.2 and the package-local VADER lexicon. The notebook therefore
checks the installed version and the lexicon checksum before sentiment analysis. It deliberately
contains no package-install or download commands: dependencies belong in the project environment,
and execution must work offline once they are installed.

A checksum match demonstrates that the supplied resource has not been replaced or altered.

In [2]:
import hashlib
import importlib.metadata
import json
import math
import re
import sys

import nltk
import numpy as np
import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer

print(f"Python version: {sys.version.split()[0]}")
print(f"pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"NLTK version: {nltk.__version__}")

required_nltk_version = "3.9.2"
if nltk.__version__ != required_nltk_version:
    raise RuntimeError(
        f"This notebook requires NLTK {required_nltk_version}, "
        f"but NLTK {nltk.__version__} is active."
    )

Python version: 3.13.9
pandas version: 2.3.3
NumPy version: 2.3.5
NLTK version: 3.9.2


In [4]:
def sha256_file(file_path):
    '''Return the SHA-256 checksum of a file.

    Parameters
    ----------
    file_path : str or pathlib.Path
        Path to the file whose bytes will be hashed.
    '''
    digest = hashlib.sha256()
    with Path(file_path).open("rb") as file_handle:
        for block in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


with RESOURCE_CONTRACT_INPUT.open("r", encoding="utf-8") as file_handle:
    resource_contract = json.load(file_handle)

expected_lexicon_checksum = resource_contract["vader_lexicon_sha256"]
observed_lexicon_checksum = sha256_file(VADER_LEXICON_INPUT)

assert observed_lexicon_checksum == expected_lexicon_checksum
assert resource_contract["nltk_version"] == nltk.__version__
print("Frozen VADER lexicon checksum and NLTK version: PASS")

# Add the supplied resource root before creating the analyser.
local_nltk_data_path = str((DATA_DIR / "resources/nltk_data").resolve())
if local_nltk_data_path not in nltk.data.path:
    nltk.data.path.insert(0, local_nltk_data_path)

sentiment_analyser = SentimentIntensityAnalyzer()

Frozen VADER lexicon checksum and NLTK version: PASS


## Task 1 - Data cleansing and explanation

### 1.0 Initial investigation and shared evidence

Before diagnosing individual records, the three batches must be loaded separately and checked
against the published structure. Separate variables preserve each batch's identity and prevent
accidental merging of the required deliverables.

The initial hashes are saved so the final cell can prove that notebook execution did not alter
the raw files. Deep snapshots provide a second, in-memory check that the source DataFrames were
not modified by later analysis.

In [5]:
source_paths = {
    "dirty": DIRTY_INPUT,
    "missing": MISSING_INPUT,
    "outlier": OUTLIER_INPUT,
    "warehouses": WAREHOUSE_INPUT,
    "manifest": MANIFEST_INPUT,
    "resource_contract": RESOURCE_CONTRACT_INPUT,
    "vader_lexicon": VADER_LEXICON_INPUT,
}

initial_source_hashes = {
    source_name: sha256_file(source_path)
    for source_name, source_path in source_paths.items()
}

dirty_source = pd.read_csv(DIRTY_INPUT)
missing_source = pd.read_csv(MISSING_INPUT)
outlier_source = pd.read_csv(OUTLIER_INPUT)
warehouses_source = pd.read_csv(WAREHOUSE_INPUT)

source_batches = {
    "dirty": dirty_source,
    "missing": missing_source,
    "outlier": outlier_source,
}

# These snapshots are used only to prove that the source DataFrames remain unchanged.
source_snapshots = {
    batch_name: dataframe.copy(deep=True)
    for batch_name, dataframe in source_batches.items()
}

print("Source files loaded without writing any output files.")

Source files loaded without writing any output files.


### 1.0.1 Structure, schema, missingness and keys

These checks establish the grain and basic quality of each batch. The specification requires
500 rows and the same 16 columns in the published order. Order IDs must be unique within and
across batches because the files contain different orders, not three versions of the same data.

Missingness is reported rather than filled. At this foundation stage, the output is evidence for
later decisions and must not change any source value.

In [6]:
overview_rows = []

for batch_name, dataframe in source_batches.items():
    overview_rows.append(
        {
            "batch": batch_name,
            "rows": len(dataframe),
            "columns": len(dataframe.columns),
            "unique_order_ids": dataframe["order_id"].nunique(),
            "unique_customers": dataframe["customer_id"].nunique(),
            "duplicate_rows": int(dataframe.duplicated().sum()),
            "missing_cells": int(dataframe.isna().sum().sum()),
        }
    )

    assert len(dataframe) == EXPECTED_ROWS_PER_BATCH
    assert list(dataframe.columns) == EXPECTED_COLUMNS
    assert dataframe["order_id"].is_unique

batch_overview = pd.DataFrame(overview_rows).set_index("batch")
display(batch_overview)

combined_order_ids = pd.concat(
    [dataframe["order_id"] for dataframe in source_batches.values()],
    ignore_index=True,
)
assert len(combined_order_ids) == 1500
assert combined_order_ids.nunique() == 1500
print("Row counts, schema, and within/across-batch order ID checks: PASS")

,rows,columns,unique_order_ids,unique_customers,duplicate_rows,missing_cells
batch,,,,,,
dirty,500,16,500,317,0,0
missing,500,16,500,312,0,81
outlier,500,16,500,318,0,0


Row counts, schema, and within/across-batch order ID checks: PASS


In [8]:
dtype_summary = pd.DataFrame(
    {
        batch_name: dataframe.dtypes.astype(str)
        for batch_name, dataframe in source_batches.items()
    }
)
missingness_summary = pd.DataFrame(
    {
        batch_name: dataframe.isna().sum()
        for batch_name, dataframe in source_batches.items()
    }
)

print("Data types")
display(dtype_summary)
print("Missing cells by field")
display(missingness_summary)

Data types


,dirty,missing,outlier
order_id,object,object,object
customer_id,object,object,object
date,object,object,object
nearest_warehouse,object,object,object
shopping_cart,object,object,object
order_price,float64,float64,float64
delivery_charges,float64,float64,float64
customer_lat,float64,float64,float64
customer_long,float64,float64,float64
coupon_discount,int64,int64,int64


Missing cells by field


,dirty,missing,outlier
order_id,0,0,0
customer_id,0,0,0
date,0,0,0
nearest_warehouse,0,11,0
shopping_cart,0,0,0
order_price,0,10,0
delivery_charges,0,10,0
customer_lat,0,0,0
customer_long,0,0,0
coupon_discount,0,0,0


In [9]:
categorical_columns = [
    "nearest_warehouse",
    "season",
    "is_expedited_delivery",
    "is_happy_customer",
    "coupon_discount",
]

for column_name in categorical_columns:
    category_counts = pd.DataFrame(
        {
            batch_name: dataframe[column_name].value_counts(dropna=False)
            for batch_name, dataframe in source_batches.items()
        }
    ).fillna(0).astype(int)
    print(f"Observed values for {column_name}")
    display(category_counts)

Observed values for nearest_warehouse


,dirty,missing,outlier
nearest_warehouse,,,
Baker's,3,0,0
Bakers,174,188,170
Nickolsen,3,0,0
Nickolson,82,84,98
THOMPSON WH,3,0,0
Thompson,235,217,232
NaN,0,11,0


Observed values for season


,dirty,missing,outlier
season,,,
Autumn,137,144,122
Spring,117,116,125
Summer,121,112,134
Winter,125,118,119
NaN,0,10,0


Observed values for is_expedited_delivery


,dirty,missing,outlier
is_expedited_delivery,,,
False,402,392,408
True,98,98,92
NaN,0,10,0


Observed values for is_happy_customer


,dirty,missing,outlier
is_happy_customer,,,
False,94,91,104
True,406,399,396
NaN,0,10,0


Observed values for coupon_discount


,dirty,missing,outlier
coupon_discount,,,
0,107,116,105
5,97,95,92
10,109,104,107
15,96,99,101
20,66,61,53
25,25,25,42


### 1.0.2 Shopping-cart parsing

`shopping_cart` is JSON, even though the CSV representation contains doubled quotation marks.
The CSV reader restores the JSON string, and `json.loads` safely parses it. `eval` is not used.

The parser below also validates the expected object structure: each entry must contain a product
name and a positive integer quantity.

In [10]:
def parse_shopping_cart(cart_text):
    '''Parse and validate one shopping-cart JSON string.

    Parameters
    ----------
    cart_text : str
        JSON text containing a list of product and quantity objects.
    '''
    if not isinstance(cart_text, str):
        raise TypeError("shopping_cart must be supplied as text")

    parsed_cart = json.loads(cart_text)
    if not isinstance(parsed_cart, list) or len(parsed_cart) == 0:
        raise ValueError("shopping_cart must be a non-empty JSON list")

    for item in parsed_cart:
        if not isinstance(item, dict):
            raise ValueError("Every cart entry must be a JSON object")
        if set(item) != {"product", "quantity"}:
            raise ValueError("Every cart entry needs product and quantity keys")
        if not isinstance(item["product"], str) or not item["product"].strip():
            raise ValueError("Every product must have a non-empty text name")
        if isinstance(item["quantity"], bool) or not isinstance(item["quantity"], int):
            raise ValueError("Every quantity must be an integer")
        if item["quantity"] <= 0:
            raise ValueError("Every quantity must be positive")

    return parsed_cart

In [11]:
cart_summary_rows = []
all_product_names = set()

for batch_name, dataframe in source_batches.items():
    parsed_carts = []
    parsing_errors = []

    for row_index, cart_text in dataframe["shopping_cart"].items():
        try:
            parsed_carts.append(parse_shopping_cart(cart_text))
        except (TypeError, ValueError, json.JSONDecodeError) as error:
            parsing_errors.append((row_index, str(error)))

    cart_lengths = [len(cart) for cart in parsed_carts]
    quantities = [item["quantity"] for cart in parsed_carts for item in cart]
    batch_products = {
        item["product"] for cart in parsed_carts for item in cart
    }
    all_product_names.update(batch_products)

    cart_summary_rows.append(
        {
            "batch": batch_name,
            "parsing_errors": len(parsing_errors),
            "distinct_products": len(batch_products),
            "minimum_cart_lines": min(cart_lengths),
            "maximum_cart_lines": max(cart_lengths),
            "minimum_quantity": min(quantities),
            "maximum_quantity": max(quantities),
        }
    )
    assert len(parsing_errors) == 0

product_names = sorted(all_product_names)
assert len(product_names) == 12

display(pd.DataFrame(cart_summary_rows).set_index("batch"))
display(pd.DataFrame({"product": product_names}))
print("All shopping carts are valid JSON with positive integer quantities: PASS")

,parsing_errors,distinct_products,minimum_cart_lines,maximum_cart_lines,minimum_quantity,maximum_quantity
batch,,,,,,
dirty,0,12,1,5,1,3
missing,0,12,1,5,1,3
outlier,0,12,1,5,1,3


,product
0,Candle Edge 392
1,Candle Edge 932
2,Candle Flux 162
3,Candle Lumen 220
4,Candle Orbit 548
5,Candle Orbit 674
6,Candle Shift 916
7,Candle Vista 110
8,Vela Core 957
9,Vela Halo 137


All shopping carts are valid JSON with positive integer quantities: PASS


### 1.0.3 Current product-price inference

Each order price creates a linear equation: the quantity of each product multiplied by its
current unit price must sum to `order_price`. The outlier batch is suitable evidence because the
specification states that only `delivery_charges` can be anomalous there; its carts and order
prices are reliable.

A reproducible 80% subset is used to infer a provisional price vector. The remaining 20% is held
out to check that the prices reproduce unseen equations. Matrix rank must equal the number of
products, otherwise the equations do not uniquely identify every price. After validation, the
final price vector is refitted using all reliable outlier-batch equations.

In [12]:
def build_cart_matrix(dataframe, product_names):
    '''Build a product-quantity matrix from an order DataFrame.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Orders containing a shopping_cart column with JSON cart text.
    product_names : sequence of str
        Ordered product names defining the matrix columns.
    '''
    product_to_column = {
        product_name: column_index
        for column_index, product_name in enumerate(product_names)
    }
    quantity_matrix = np.zeros(
        (len(dataframe), len(product_names)),
        dtype=float,
    )

    for matrix_row, cart_text in enumerate(dataframe["shopping_cart"]):
        parsed_cart = parse_shopping_cart(cart_text)
        for item in parsed_cart:
            product_name = item["product"]
            if product_name not in product_to_column:
                raise ValueError(f"Unknown product in cart: {product_name}")
            matrix_column = product_to_column[product_name]
            quantity_matrix[matrix_row, matrix_column] += item["quantity"]

    return quantity_matrix


def infer_unit_prices(dataframe, product_names):
    '''Infer unit prices from reliable cart and order-price equations.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Reliable orders with complete shopping_cart and order_price values.
    product_names : sequence of str
        Ordered product names whose prices will be estimated.
    '''
    if dataframe["order_price"].isna().any():
        raise ValueError("Price inference requires complete order_price values")

    quantity_matrix = build_cart_matrix(dataframe, product_names)
    observed_order_prices = dataframe["order_price"].to_numpy(dtype=float)
    estimated_prices, residuals, matrix_rank, singular_values = np.linalg.lstsq(
        quantity_matrix,
        observed_order_prices,
        rcond=None,
    )

    if matrix_rank != len(product_names):
        raise ValueError(
            "The cart equations do not have full rank, so prices are not uniquely identified"
        )

    rounded_prices = np.round(estimated_prices, 2)
    price_series = pd.Series(
        rounded_prices,
        index=product_names,
        name="unit_price_aud",
    )
    diagnostics = {
        "number_of_equations": len(dataframe),
        "number_of_products": len(product_names),
        "matrix_rank": int(matrix_rank),
        "least_squares_residual_sum": (
            float(residuals[0]) if len(residuals) else 0.0
        ),
        "smallest_singular_value": float(singular_values.min()),
    }
    return price_series, diagnostics

In [13]:
random_generator = np.random.default_rng(RANDOM_SEED)
shuffled_positions = random_generator.permutation(len(outlier_source))
training_size = int(0.80 * len(outlier_source))

training_positions = shuffled_positions[:training_size]
validation_positions = shuffled_positions[training_size:]
price_training_data = outlier_source.iloc[training_positions].copy(deep=True)
price_validation_data = outlier_source.iloc[validation_positions].copy(deep=True)

provisional_unit_prices, training_diagnostics = infer_unit_prices(
    price_training_data,
    product_names,
)

validation_cart_matrix = build_cart_matrix(
    price_validation_data,
    product_names,
)
validation_predictions = np.round(
    validation_cart_matrix @ provisional_unit_prices.to_numpy(),
    2,
)
validation_observed = price_validation_data["order_price"].to_numpy(dtype=float)
validation_absolute_errors = np.abs(
    validation_predictions - validation_observed
)

assert training_diagnostics["matrix_rank"] == len(product_names)
assert validation_absolute_errors.max() <= MONEY_TOLERANCE

display(pd.Series(training_diagnostics, name="training_diagnostic"))
display(
    pd.Series(
        {
            "held_out_equations": len(price_validation_data),
            "maximum_absolute_error_aud": validation_absolute_errors.max(),
            "mean_absolute_error_aud": validation_absolute_errors.mean(),
        },
        name="held_out_validation",
    )
)
print("Held-out cart-equation validation: PASS")

number_of_equations           4.000000e+02
number_of_products            1.200000e+01
matrix_rank                   1.200000e+01
least_squares_residual_sum    6.633966e-22
smallest_singular_value       9.212117e+00
Name: training_diagnostic, dtype: float64

held_out_equations            100.0
maximum_absolute_error_aud      0.0
mean_absolute_error_aud         0.0
Name: held_out_validation, dtype: float64

Held-out cart-equation validation: PASS


In [15]:
final_unit_prices, final_price_diagnostics = infer_unit_prices(
    outlier_source,
    product_names,
)

display(final_unit_prices.to_frame())
display(pd.Series(final_price_diagnostics, name="final_fit_diagnostic"))

price_check_rows = []
for batch_name, dataframe in source_batches.items():
    observed_mask = dataframe["order_price"].notna()
    observed_orders = dataframe.loc[observed_mask]
    cart_matrix = build_cart_matrix(observed_orders, product_names)
    calculated_prices = np.round(
        cart_matrix @ final_unit_prices.to_numpy(),
        2,
    )
    observed_prices = observed_orders["order_price"].to_numpy(dtype=float)
    absolute_errors = np.abs(calculated_prices - observed_prices)

    price_check_rows.append(
        {
            "batch": batch_name,
            "observed_order_prices": len(observed_orders),
            "matches_within_0.01": int(
                (absolute_errors <= MONEY_TOLERANCE).sum()
            ),
            "values_to_investigate_later": int(
                (absolute_errors > MONEY_TOLERANCE).sum()
            ),
            "maximum_absolute_error_aud": absolute_errors.max(),
        }
    )

display(pd.DataFrame(price_check_rows).set_index("batch"))

,unit_price_aud
Candle Edge 392,375.75
Candle Edge 932,1490.48
Candle Flux 162,1060.29
Candle Lumen 220,3529.39
Candle Orbit 548,256.07
Candle Orbit 674,324.06
Candle Shift 916,1858.77
Candle Vista 110,1816.59
Vela Core 957,271.47
Vela Halo 137,759.89


number_of_equations           5.000000e+02
number_of_products            1.200000e+01
matrix_rank                   1.200000e+01
least_squares_residual_sum    1.367808e-21
smallest_singular_value       1.038364e+01
Name: final_fit_diagnostic, dtype: float64

,observed_order_prices,matches_within_0.01,values_to_investigate_later,maximum_absolute_error_aud
batch,,,,
dirty,500,492,8,121.75
missing,490,490,0,0.00
outlier,500,500,0,0.00


### 1.0.4 Reusable calculations for dates, seasons and order arithmetic

Dates may appear in year-first or slash-formatted day/month/year forms. The parser applies the
specification explicitly rather than asking pandas to guess. Seasons are then derived from the
calendar month using Australian seasons.

Order-price calculation follows the required item-level rounding. Order total applies the coupon
to merchandise only, adds delivery afterwards, and rounds the final result to two decimals.

In [16]:
def calculate_order_price(cart, unit_prices):
    '''Calculate a cart's merchandise price using the published rounding rule.

    Parameters
    ----------
    cart : str or list of dict
        Shopping-cart JSON text or an already parsed list of cart entries.
    unit_prices : mapping
        Mapping from each product name to its current unit price in AUD.
    '''
    parsed_cart = parse_shopping_cart(cart) if isinstance(cart, str) else cart
    item_amounts = []

    for item in parsed_cart:
        product_name = item["product"]
        if product_name not in unit_prices:
            raise KeyError(f"No unit price is available for {product_name}")
        item_amount = round(
            item["quantity"] * float(unit_prices[product_name]),
            2,
        )
        item_amounts.append(item_amount)

    return round(sum(item_amounts), 2)


def parse_order_date(date_value):
    '''Parse an order date using the explicit date rules.

    Parameters
    ----------
    date_value : str or date-like value
        Source order date. Slash-formatted numeric dates are day-first unless
        they begin with a four-digit year, which remains year-first.
    '''
    date_text = str(date_value).strip()

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", date_text):
        return pd.to_datetime(date_text, format="%Y-%m-%d")
    if re.fullmatch(r"\d{4}/\d{2}/\d{2}", date_text):
        return pd.to_datetime(date_text, format="%Y/%m/%d")
    if re.fullmatch(r"\d{1,2}/\d{1,2}/\d{4}", date_text):
        return pd.to_datetime(date_text, dayfirst=True)

    return pd.to_datetime(date_text, errors="raise")


def get_australian_season(order_date):
    '''Return the Australian season for an order date.

    Parameters
    ----------
    order_date : str or date-like value
        Order date whose calendar month determines the season.
    '''
    parsed_date = parse_order_date(order_date)
    month_number = parsed_date.month

    if month_number in (12, 1, 2):
        return "Summer"
    if month_number in (3, 4, 5):
        return "Autumn"
    if month_number in (6, 7, 8):
        return "Winter"
    return "Spring"


def calculate_order_total(order_price, coupon_discount, delivery_charge):
    '''Calculate the final order total using the published formula.

    Parameters
    ----------
    order_price : float
        Merchandise price before discount and delivery, in AUD.
    coupon_discount : float
        Percentage discount applied to merchandise only.
    delivery_charge : float
        Delivery charge added after the merchandise discount, in AUD.
    '''
    discounted_fraction = 1.0 - (float(coupon_discount) / 100.0)
    return round(
        float(order_price) * discounted_fraction + float(delivery_charge),
        2,
    )

In [17]:
assert parse_order_date("19/10/2019") == pd.Timestamp("2019-10-19")
assert parse_order_date("2019/11/09") == pd.Timestamp("2019-11-09")
assert parse_order_date("Apr 24, 2019") == pd.Timestamp("2019-04-24")

season_examples = {
    "2019-01-15": "Summer",
    "2019-03-01": "Autumn",
    "2019-06-30": "Winter",
    "2019-09-01": "Spring",
    "2019-12-31": "Summer",
}
for example_date, expected_season in season_examples.items():
    assert get_australian_season(example_date) == expected_season

assert calculate_order_total(100.00, 10, 8.00) == 98.00

example_cart = '[{"product":"Candle Edge 392","quantity":2}]'
expected_example_price = round(2 * final_unit_prices["Candle Edge 392"], 2)
assert calculate_order_price(example_cart, final_unit_prices) == expected_example_price
print("Date, season, order-price and order-total function tests: PASS")

Date, season, order-price and order-total function tests: PASS


### 1.0.5 Reusable warehouse and distance calculations

Customer-to-warehouse distance is the straight-line great-circle distance calculated with the
haversine formula and an Earth radius of 6,371 km. The nearest warehouse is the warehouse with
the minimum calculated distance. Road-routing data and shortest-path algorithms are not relevant.

Calculations retain coordinate precision; reconstructed distances are rounded to four decimals
only when returned for comparison or future repair.

In [18]:
def haversine_distance(lat1, lon1, lat2, lon2):
    '''Calculate great-circle distance between two coordinates in kilometres.

    Parameters
    ----------
    lat1 : float
        Latitude of the first point in decimal degrees.
    lon1 : float
        Longitude of the first point in decimal degrees.
    lat2 : float
        Latitude of the second point in decimal degrees.
    lon2 : float
        Longitude of the second point in decimal degrees.
    '''
    latitude_1 = math.radians(float(lat1))
    longitude_1 = math.radians(float(lon1))
    latitude_2 = math.radians(float(lat2))
    longitude_2 = math.radians(float(lon2))

    latitude_difference = latitude_2 - latitude_1
    longitude_difference = longitude_2 - longitude_1

    haversine_value = (
        math.sin(latitude_difference / 2.0) ** 2
        + math.cos(latitude_1)
        * math.cos(latitude_2)
        * math.sin(longitude_difference / 2.0) ** 2
    )
    central_angle = 2.0 * math.asin(math.sqrt(haversine_value))
    return EARTH_RADIUS_KM * central_angle


def find_nearest_warehouse(customer_lat, customer_long, warehouses):
    '''Find the closest supplied warehouse to a customer coordinate.

    Parameters
    ----------
    customer_lat : float
        Customer latitude in decimal degrees.
    customer_long : float
        Customer longitude in decimal degrees.
    warehouses : pandas.DataFrame
        Reference data containing warehouse, latitude and longitude columns.
    '''
    required_warehouse_columns = {"warehouse", "latitude", "longitude"}
    if not required_warehouse_columns.issubset(warehouses.columns):
        raise ValueError("Warehouse reference data has missing required columns")

    distances = []
    for warehouse_row in warehouses.itertuples(index=False):
        distance_km = haversine_distance(
            customer_lat,
            customer_long,
            warehouse_row.latitude,
            warehouse_row.longitude,
        )
        distances.append((warehouse_row.warehouse, distance_km))

    nearest_name, nearest_distance = min(
        distances,
        key=lambda warehouse_distance: warehouse_distance[1],
    )
    return nearest_name, round(nearest_distance, 4)

In [19]:
assert haversine_distance(-37.8, 145.0, -37.8, 145.0) == 0.0

for warehouse_row in warehouses_source.itertuples(index=False):
    calculated_name, calculated_distance = find_nearest_warehouse(
        warehouse_row.latitude,
        warehouse_row.longitude,
        warehouses_source,
    )
    assert calculated_name == warehouse_row.warehouse
    assert calculated_distance == 0.0

example_order = outlier_source.iloc[0]
example_warehouse, example_distance = find_nearest_warehouse(
    example_order["customer_lat"],
    example_order["customer_long"],
    warehouses_source,
)
print(f"Example calculated warehouse: {example_warehouse}")
print(f"Example calculated distance: {example_distance:.4f} km")
print("Haversine and nearest-warehouse function tests: PASS")

Example calculated warehouse: Nickolson
Example calculated distance: 7.2270 km
Haversine and nearest-warehouse function tests: PASS


### 1.0.6 Reusable VADER happiness classification

The supplied review must be scored exactly as written. A customer is happy when the VADER
compound score is greater than or equal to 0.05. The exact text `No review submitted` is a
documented sentinel and must return `True` without being treated as missing.

A small dummy analyser is used to test the inclusive boundary exactly. This avoids relying on a
particular sentence to happen to receive a compound score of precisely 0.05.

In [20]:
def classify_customer_happiness(review_text, analyser):
    '''Classify prior-review happiness using the assignment's VADER rule.

    Parameters
    ----------
    review_text : str
        Supplied English review text or the exact no-review sentinel.
    analyser : nltk.sentiment.vader.SentimentIntensityAnalyzer
        Configured sentiment analyser using the frozen local lexicon.
    '''
    if review_text == NO_REVIEW_SENTINEL:
        return True
    if not isinstance(review_text, str):
        raise TypeError("latest_customer_review must be text")

    compound_score = analyser.polarity_scores(review_text)["compound"]
    return bool(compound_score >= resource_contract["compound_threshold"])


class BoundaryTestAnalyser:
    '''Return a fixed compound score for an exact threshold test.'''

    def polarity_scores(self, review_text):
        '''Return a fixed VADER-like score dictionary.

        Parameters
        ----------
        review_text : str
            Review text accepted to match the real analyser interface.
        '''
        return {"compound": 0.05}


assert classify_customer_happiness(
    NO_REVIEW_SENTINEL,
    sentiment_analyser,
) is True
assert classify_customer_happiness(
    "Boundary test",
    BoundaryTestAnalyser(),
) is True
assert classify_customer_happiness(
    "This product is excellent and reliable.",
    sentiment_analyser,
) is True
assert classify_customer_happiness(
    "This product is terrible and disappointing.",
    sentiment_analyser,
) is False
print("VADER resource, sentinel and inclusive-threshold tests: PASS")

VADER resource, sentinel and inclusive-threshold tests: PASS


### 1.0.7 Final foundation checks and stage boundary

The final checks repeat the source hashes and compare every source DataFrame with its deep
snapshot. A passing result confirms that the investigation and demonstrations did not modify the
original files or the in-memory source tables.

This notebook intentionally stops here. Later notebooks or sections may call these functions on
explicit working copies to diagnose dirty fields, recover missing values and model delivery-charge
residuals. No correction should be made merely because one dependent consistency check fails.

In [22]:
final_source_hashes = {
    source_name: sha256_file(source_path)
    for source_name, source_path in source_paths.items()
}
assert final_source_hashes == initial_source_hashes

for batch_name, source_dataframe in source_batches.items():
    pd.testing.assert_frame_equal(
        source_dataframe,
        source_snapshots[batch_name],
        check_dtype=True,
        check_exact=True,
    )

cleaning_output_names = [
    f"{GROUP_ID}_dirty_clean.csv",
    f"{GROUP_ID}_missing_clean.csv",
    f"{GROUP_ID}_outlier_clean.csv",
    "audit_log.csv",
]
created_cleaning_outputs = [
    output_name
    for output_name in cleaning_output_names
    if Path(output_name).exists()
]
assert created_cleaning_outputs == []

final_summary = pd.Series(
    {
        "source_file_hashes_unchanged": True,
        "source_dataframes_unchanged": True,
        "cleaning_outputs_created": False,
        "batches_checked": len(source_batches),
        "distinct_order_ids": combined_order_ids.nunique(),
        "products_identified": len(product_names),
        "price_matrix_rank": final_price_diagnostics["matrix_rank"],
        "vader_contract_verified": True,
    },
    name="shared_foundation_result",
)
display(final_summary)

source_file_hashes_unchanged     True
source_dataframes_unchanged      True
cleaning_outputs_created        False
batches_checked                     3
distinct_order_ids               1500
products_identified                12
price_matrix_rank                  12
vader_contract_verified          True
Name: shared_foundation_result, dtype: object